# EXP02: ORB max_hold + KC x ORB 组合网格

**合并**: NB02 + NB03

1. ORB max_hold 扫描 (7 变体)
2. KC x ORB max_hold 交叉网格 (4x4 = 16 变体)

In [ ]:
import sys; sys.path.insert(0, '..')
from backtest import DataBundle, run_variant, C12_KWARGS
from backtest.stats import print_ranked
import config
import pandas as pd

data = DataBundle.load_default()

LIVE_KWARGS = {
    **C12_KWARGS,
    "intraday_adaptive": True,
    "choppy_threshold": 0.35,
    "kc_only_threshold": 0.60,
    "spread_cost": 0.50,
}
print('Data loaded')

## Part 1: ORB max_hold 扫描

In [ ]:
orb_results = []
for mh in [4, 6, 8, 12, 16, 24, 32]:
    r = run_variant(data, f"ORB_mh={mh}", orb_max_hold_m15=mh, **LIVE_KWARGS)
    r['orb_max_hold'] = mh
    orb_results.append(r)

print_ranked(orb_results)

In [ ]:
for r in orb_results:
    orb_trades = [t for t in r['_trades'] if t.strategy == 'orb']
    df = pd.DataFrame([{'exit_reason': t.exit_reason, 'pnl': t.pnl, 'bars_held': t.bars_held} for t in orb_trades])
    print(f"\n=== ORB max_hold={r['orb_max_hold']} ({len(orb_trades)} ORB trades) ===")
    if len(df) > 0:
        print(df.groupby('exit_reason')['pnl'].agg(['sum', 'mean', 'count']))

## Part 2: KC x ORB max_hold 交叉网格

In [ ]:
original_mh = config.STRATEGIES['keltner']['max_hold_bars']
combo_results = []

for kc_mh in [8, 12, 16, 20]:
    for orb_mh in [8, 12, 16, 24]:
        config.STRATEGIES['keltner']['max_hold_bars'] = kc_mh
        r = run_variant(data, f"KC{kc_mh}_ORB{orb_mh}", orb_max_hold_m15=orb_mh, **LIVE_KWARGS)
        r['kc_max_hold'] = kc_mh
        r['orb_max_hold'] = orb_mh
        combo_results.append(r)

config.STRATEGIES['keltner']['max_hold_bars'] = original_mh
print_ranked(combo_results)

In [ ]:
import json
from backtest.runner import sanitize_for_json
with open('../data/exp02_results.json', 'w') as f:
    json.dump(sanitize_for_json(orb_results + combo_results), f, indent=2)
print('Saved to data/exp02_results.json')